In [ ]:
import sys, os
sys.path.append('./src/')


from PIL import Image
from VAE_trainers import EpochPyroTrainer, AdversarialEpochPyroTrainer, ThresholdPyroTrainer, AdversarialThresholdPyroTrainer
from CNN_variants import CNNVAE, CNNCVAE, CNNCSVAENA, CNNCSVAE, CNNHCSVAENA, CNNHCSVAE, CNNSDIVA, CNNCCVAE, CNNDLVAE
from matplotlib.colors import LinearSegmentedColormap
from tqdm import tqdm, trange
from umap import UMAP
from torchvision.datasets import CelebA, MNIST
from torchvision.transforms import ToTensor


import torch, pyro
import numpy as np
import matplotlib.pyplot as plt
import copy, cv2
import pyro.optim as opt
import pandas as pd

np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)


# Subset wrapper for attributes
class SubsetWrapper(torch.utils.data.Dataset):
    
    def __init__(self, base_dataset, idxs):
        self.base_dataset = base_dataset
        self.idxs = idxs

    def __getitem__(self, idx):

        if isinstance(idx, np.ndarray):
            elems = list(np.array(self.idxs)[idx])

        else:
            elems = self.idxs[idx]
        
        if isinstance(elems, int):
            im, lab = self.base_dataset[elems]
            lab = lab.unsqueeze(dim=0)


            
        
        else:
            im, lab = [], []
            for idx in elems:
                c_im, c_lab = self.base_dataset[idx]
                im.append(c_im)
                lab.append(c_lab)
             
             
            im, lab = torch.stack(im), torch.vstack(lab)
            
        return im, lab, lab 

    def __len__(self):
        return len(self.idxs)
        
hex_colors = ["#F23E2E", "#5888A6"]
cmap = LinearSegmentedColormap.from_list("cmap", hex_colors)

## Resize Imgs (run only to resize / preprocess images, rename dirs manually after running)

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from PIL import Image
from tqdm import tqdm
import os
from skimage import transform, filters, color


def sample_img_augment_params(translation_sigma=1.0, scale_sigma=0.01,
                              rotation_sigma=0.01, gamma_sigma=0.07,
                              contrast_sigma=0.07, hue_sigma=0.0125):
    translation = np.random.normal(scale=translation_sigma, size=2)
    scale = np.random.normal(loc=1.0, scale=scale_sigma)
    rotation = np.random.normal(scale=rotation_sigma)
    mu = gamma_sigma**2
    gamma = np.random.normal(loc=mu, scale=gamma_sigma)
    gamma = np.exp(gamma/np.log(2))
    mu = contrast_sigma**2
    contrast = np.random.normal(loc=mu, scale=contrast_sigma)
    contrast = np.exp(contrast/np.log(2))
    hue = np.random.normal(scale=hue_sigma)
    return translation, scale, rotation, gamma, contrast, hue
    

def img_augment(img, translation=0.0, scale=1.0, rotation=0.0, gamma=1.0,
                contrast=1.0, hue=0.0, border_mode='constant'):
    if not (np.all(np.isclose(translation, [0.0, 0.0])) and
            np.isclose(scale, 1.0) and
            np.isclose(rotation, 0.0)):
        img_center = np.array(img.shape[:2]) / 2.0
        scale = (scale, scale)
        transf = transform.SimilarityTransform(translation=-img_center)
        transf += transform.SimilarityTransform(scale=scale, rotation=rotation)
        translation = img_center + translation
        transf += transform.SimilarityTransform(translation=translation)
        img = transform.warp(img, transf, order=3, mode=border_mode)
    if not np.isclose(gamma, 1.0):
        img **= gamma
    colorspace = 'rgb'
    if not np.isclose(contrast, 1.0):
        img = color.convert_colorspace(img, colorspace, 'hsv')
        colorspace = 'hsv'
        img[..., 1:] **= contrast
    if not np.isclose(hue, 0.0):
        img = color.convert_colorspace(img, colorspace, 'hsv')
        colorspace = 'hsv'
        img[..., 0] += hue
        img[img[..., 0] > 1.0, 0] -= 1.0
        img[img[..., 0] < 0.0, 0] += 1.0
    img = color.convert_colorspace(img, colorspace, 'rgb')
    if np.min(img) < 0.0 or np.max(img) > 1.0:
        raise ValueError('Invalid values in output image.')
    return img


def _resize(args):
    img, rescale_size, bbox = args
    img = img[bbox[0]:bbox[1], bbox[2]:bbox[3]]
    # Smooth image before resize to avoid moire patterns
    scale = img.shape[0] / float(rescale_size)
    sigma = np.sqrt(scale) / 2.0
    img = filters.gaussian(img, sigma=sigma)
    img = transform.resize(img, (rescale_size, rescale_size, 3), order=3)
    img = (img*255).astype(np.uint8)
    return img


def _resize_augment(args):
    img, rescale_size, bbox = args
    augment_params = sample_img_augment_params(
        translation_sigma=2.00, scale_sigma=0.01, rotation_sigma=0.01,
        gamma_sigma=0.05, contrast_sigma=0.05, hue_sigma=0.01
    )
    img = img_augment(img, *augment_params, border_mode='constant')
    img = _resize((img, rescale_size, bbox))
    return img


img_size=64 
bbox=(40, 218-30, 15, 178-15)


# Define paths
src_dir = 'data/celeba/img_align_celeba/'  # The dataset needs to be downloaded separately
dst_dir = 'data/celeba/img_align_celeba_64_aug'  # Rename the outputted folder after conversion for correct loading!

# Ensure output directory exists
os.makedirs(dst_dir, exist_ok=True)

# List of filenames
base_im_dir = os.listdir(src_dir)

# Function to resize and save one image
def resize_and_save(fname):
    src_path = os.path.join(src_dir, fname)
    dst_path = os.path.join(dst_dir, fname)
    try:
        im = np.array(Image.open(src_path)) / 255
        im = _resize_augment([im, img_size, bbox])
        Image.fromarray(im).save(dst_path)

    except Exception as e:
        return f"Failed on {fname}: {e}"

# Parallel execution with progress bar
with ProcessPoolExecutor() as executor:
    list(tqdm(executor.map(resize_and_save, base_im_dir), total=len(base_im_dir)))


## Data (run after setting up data dir)

In [ ]:
splits = pd.read_csv('./data/celeba/list_eval_partition.txt', sep=' ', header=None)
splits.columns = ['fname', 'split']
splits['split'] = splits['split'].astype('category')
splits['split'] = splits['split'].cat.rename_categories(['train', 'test', 'eval'])

In [ ]:
attrs = pd.read_csv('./data/celeba/list_attr_celeba.txt', header=1, skipinitialspace=True, sep=' ')
attrs.columns = ['fname'] + list(attrs.columns)[:-1]

attrs

In [ ]:
attr_pd = pd.read_csv('./data/celeba/list_attr_celeba.txt', header=1)

In [ ]:
def glasses_transform(target):
    return target[15].type(torch.float32)

dataset_glasses = CelebA(root="./data", download=False, transform=ToTensor() ,target_transform=glasses_transform, split='all')

In [ ]:
batch_size=64
num_workers=16

np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)


## Training data creation

attrs_train = attrs[attrs['fname'].isin(splits[splits['split'] == 'train']['fname'])]
glasses_train_idx =  attrs_train[attrs_train['Eyeglasses'] == 1].index
glasses_train_count = len(glasses_train_idx)
glasses_negative_train_idx = attrs_train[attrs_train['Eyeglasses'] == -1].sample(2*glasses_train_count).index
glasses_train_idx = list(glasses_train_idx) + list(glasses_negative_train_idx)
train_set = SubsetWrapper(dataset_glasses, glasses_train_idx)


## Test data creation
attrs_test = attrs[attrs['fname'].isin(splits[splits['split'] == 'test']['fname'])]
glasses_test_idx =  attrs_test[attrs_test['Eyeglasses'] == 1].index
glasses_test_count = len(glasses_test_idx)
glasses_negative_test_idx = attrs_test[attrs_test['Eyeglasses'] == -1].sample(2*glasses_test_count).index
glasses_test_idx = list(glasses_test_idx) + list(glasses_negative_test_idx)

test_set = SubsetWrapper(dataset_glasses, glasses_test_idx)


## Set loaders
train_loader, test_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, num_workers=num_workers, timeout=100), torch.utils.data.DataLoader(test_set, batch_size=batch_size, num_workers=num_workers, timeout=100)

## CSVAENA

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

csvaena = CNNCSVAENA((64,64), 3, [1], latent_dim=2048, w_dim=2, channels=[64,128,256], repeats=[2,1,1], cnn_arch='conv+pool', num_layers=0, recon_weight=1, z_kl_weight=1e-4, kernel_size=5)
csvaena_trainer = ThresholdPyroTrainer(0, 50, csvaena, train_loader, test_loader, opt.AdamW({"lr": 1e-4}))
csvaena_trainer.train()

In [ ]:
preds_glasses = csvaena_trainer.predictive(*csvaena_trainer._send_args_to_device(csvaena_trainer.test_loader.dataset[:100], csvaena_trainer.device))
z_s_glasses = preds_glasses['z'][0].cpu()
w_s_glasses = preds_glasses['w'][0].cpu()
recons_glasses = preds_glasses['rec'][0, 0].cpu()


preds_ng = csvaena_trainer.predictive(*csvaena_trainer._send_args_to_device(csvaena_trainer.test_loader.dataset[glasses_test_count:glasses_test_count+100], csvaena_trainer.device))
z_s_ng = preds_ng['z'][0].cpu() 
w_s_ng = preds_ng['w'][0].cpu() 
recons_ng = preds_ng['rec'][0, 0].cpu()

z_s = torch.vstack((z_s_glasses, z_s_ng))
w_s = torch.vstack((w_s_glasses, w_s_ng))
recons = torch.vstack((recons_glasses, recons_ng))
y_s = torch.vstack((csvaena_trainer.test_loader.dataset[:100][1], csvaena_trainer.test_loader.dataset[glasses_test_count:glasses_test_count+100][1]))
origs = torch.vstack((csvaena_trainer.test_loader.dataset[:100][0], csvaena_trainer.test_loader.dataset[glasses_test_count:glasses_test_count+100][0]))

In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

for i in range(10):
    im, label = test_set[i][0], test_set[i][1]
    im = np.einsum('ijk -> jki', im)

    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


for i in range(10, 20):
    im, label = test_set[glasses_test_count + i][0], test_set[glasses_test_count + i][1]
    im = np.einsum('ijk -> jki', im)


    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

for i in range(10):
    im, label = recons[i], test_set[i][1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


for i in range(10, 20):
    im, label = recons[100 + i], test_set[glasses_test_count + i][1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


## CSVAE

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)


csvae = CNNCSVAE((64,64), 3, [1], latent_dim=2048, w_dim=2, channels=[64,128,256], repeats=[2,1,1], cnn_arch='conv+pool', num_layers=0, recon_weight=1e3, z_kl_weight=1e-4, kernel_size=5)
csvae_trainer = AdversarialThresholdPyroTrainer(0, 50, 1, 1, csvae, train_loader, test_loader, opt.AdamW({"lr": 1e-4}))
csvae_trainer.train()

In [ ]:
preds_glasses = csvae_trainer.predictive(*csvae_trainer._send_args_to_device(csvae_trainer.test_loader.dataset[:100], csvae_trainer.device))
z_s_glasses = preds_glasses['z'][0].cpu()
w_s_glasses = preds_glasses['w'][0].cpu()
recons_glasses = preds_glasses['rec'][0, 0].cpu()


preds_ng = csvae_trainer.predictive(*csvae_trainer._send_args_to_device(csvae_trainer.test_loader.dataset[glasses_test_count:glasses_test_count+100], csvae_trainer.device))
z_s_ng = preds_ng['z'][0].cpu() 
w_s_ng = preds_ng['w'][0].cpu() 
recons_ng = preds_ng['rec'][0, 0].cpu()

z_s = torch.vstack((z_s_glasses, z_s_ng))
w_s = torch.vstack((w_s_glasses, w_s_ng))
recons = torch.vstack((recons_glasses, recons_ng))
y_s = torch.vstack((csvae_trainer.test_loader.dataset[:100][1], csvae_trainer.test_loader.dataset[glasses_test_count:glasses_test_count+100][1]))
origs = torch.vstack((csvae_trainer.test_loader.dataset[:100][0], csvae_trainer.test_loader.dataset[glasses_test_count:glasses_test_count+100][0]))

In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

for i in range(10):
    im, label = test_set[i][0], test_set[i][1]
    im = np.einsum('ijk -> jki', im)

    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


for i in range(10, 20):
    im, label = test_set[glasses_test_count + i][0], test_set[glasses_test_count + i][1]
    im = np.einsum('ijk -> jki', im)


    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')

In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

for i in range(10):
    im, label = recons[i], test_set[i][1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


for i in range(10, 20):
    im, label = recons[100 + i], test_set[glasses_test_count + i][1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


## HCSVAENA

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

hcsvaena = CNNHCSVAENA((64,64), 3, [1], latent_dim=2048, w_dim=2, channels=[64,128,256], repeats=[2,1,1], cnn_arch='conv+pool', num_layers=0, recon_weight=1e3, z_kl_weight=1e-4, kernel_size=5)
hcsvaena_trainer = ThresholdPyroTrainer(0, 50, hcsvaena, train_loader, test_loader, opt.AdamW({"lr": 1e-4}))
hcsvaena_trainer.train()

In [ ]:
preds_glasses = hcsvaena_trainer.predictive(*hcsvaena_trainer._send_args_to_device(hcsvaena_trainer.test_loader.dataset[:100], hcsvaena_trainer.device))
z_s_glasses = preds_glasses['z'][0].cpu()
w_s_glasses = preds_glasses['w'][0].cpu()
recons_glasses = preds_glasses['rec'][0, 0].cpu()


preds_ng = hcsvaena_trainer.predictive(*hcsvaena_trainer._send_args_to_device(hcsvaena_trainer.test_loader.dataset[glasses_test_count:glasses_test_count+100], hcsvaena_trainer.device))
z_s_ng = preds_ng['z'][0].cpu() 
w_s_ng = preds_ng['w'][0].cpu() 
recons_ng = preds_ng['rec'][0, 0].cpu()

z_s = torch.vstack((z_s_glasses, z_s_ng))
w_s = torch.vstack((w_s_glasses, w_s_ng))
recons = torch.vstack((recons_glasses, recons_ng))
y_s = torch.vstack((hcsvaena_trainer.test_loader.dataset[:100][1], hcsvaena_trainer.test_loader.dataset[glasses_test_count:glasses_test_count+100][1]))
y_s = torch.vstack((hcsvaena_trainer.test_loader.dataset[:100][0], hcsvaena_trainer.test_loader.dataset[glasses_test_count:glasses_test_count+100][0]))

In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

for i in range(10):
    im, label = test_set[i][0], test_set[i][1]
    im = np.einsum('ijk -> jki', im)

    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


for i in range(10, 20):
    im, label = test_set[glasses_test_count + i][0], test_set[glasses_test_count + i][1]
    im = np.einsum('ijk -> jki', im)


    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')

In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

for i in range(10):
    im, label = recons[i], test_set[i][1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


for i in range(10, 20):
    im, label = recons[100 + i], test_set[glasses_test_count + i][1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


## HCSVAE

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

hcsvae = CNNHCSVAE((64,64), 3, [1], latent_dim=2048, w_dim=2, channels=[64,128,256], repeats=[2,1,1], cnn_arch='conv+pool', num_layers=0, recon_weight=1e4, z_kl_weight=1e-4, kernel_size=5)
hcsvae_trainer = AdversarialThresholdPyroTrainer(0, 50, 1, 1, hcsvae, train_loader, test_loader, opt.AdamW({"lr": 1e-4}))
hcsvae_trainer.train()

In [ ]:
preds_glasses = hcsvae_trainer.predictive(*hcsvae_trainer._send_args_to_device(hcsvae_trainer.test_loader.dataset[:100], hcsvae_trainer.device))
z_s_glasses = preds_glasses['z'][0].cpu()
w_s_glasses = preds_glasses['w'][0].cpu()
recons_glasses = preds_glasses['rec'][0, 0].cpu()


preds_ng = hcsvae_trainer.predictive(*hcsvae_trainer._send_args_to_device(hcsvae_trainer.test_loader.dataset[glasses_test_count:glasses_test_count+100], hcsvae_trainer.device))
z_s_ng = preds_ng['z'][0].cpu() 
w_s_ng = preds_ng['w'][0].cpu() 
recons_ng = preds_ng['rec'][0, 0].cpu()

z_s = torch.vstack((z_s_glasses, z_s_ng))
w_s = torch.vstack((w_s_glasses, w_s_ng))
recons = torch.vstack((recons_glasses, recons_ng))
y_s = torch.vstack((hcsvae_trainer.test_loader.dataset[:100][1], hcsvae_trainer.test_loader.dataset[glasses_test_count:glasses_test_count+100][1]))
origs = torch.vstack((hcsvae_trainer.test_loader.dataset[:100][0], hcsvae_trainer.test_loader.dataset[glasses_test_count:glasses_test_count+100][0]))

In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

for i in range(10):
    im, label = test_set[i][0], test_set[i][1]
    im = np.einsum('ijk -> jki', im)

    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


for i in range(10, 20):
    im, label = test_set[glasses_test_count + i][0], test_set[glasses_test_count + i][1]
    im = np.einsum('ijk -> jki', im)


    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')

In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

for i in range(10):
    im, label = recons[i], test_set[i][1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


for i in range(10, 20):
    im, label = recons[100 + i], test_set[glasses_test_count + i][1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


## DIVA

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

diva = CNNSDIVA((64,64), 3, [1], latent_dim=2048, w_dim=2, channels=[64,128,256], repeats=[2,1,1], cnn_arch='conv+pool', num_layers=0, recon_weight=1e5, kl_weight=1e-4, kernel_size=5)
diva_trainer = ThresholdPyroTrainer(0, 50, diva, train_loader, test_loader, opt.AdamW({"lr": 1e-4}))
diva_trainer.train()

In [ ]:
preds_glasses = diva_trainer.predictive(*diva_trainer._send_args_to_device(diva_trainer.test_loader.dataset[:100], diva_trainer.device))
z_s_glasses = preds_glasses['z'][0].cpu()
w_s_glasses = preds_glasses['w'][0].cpu()
recons_glasses = preds_glasses['rec'][0, 0].cpu()


preds_ng = diva_trainer.predictive(*diva_trainer._send_args_to_device(diva_trainer.test_loader.dataset[glasses_test_count:glasses_test_count+100], diva_trainer.device))
z_s_ng = preds_ng['z'][0].cpu() 
w_s_ng = preds_ng['w'][0].cpu() 
recons_ng = preds_ng['rec'][0, 0].cpu()

z_s = torch.vstack((z_s_glasses, z_s_ng))
w_s = torch.vstack((w_s_glasses, w_s_ng))
recons = torch.vstack((recons_glasses, recons_ng))
y_s = torch.vstack((diva_trainer.test_loader.dataset[:100][1], diva_trainer.test_loader.dataset[glasses_test_count:glasses_test_count+100][1]))

In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

for i in range(10):
    im, label = test_set[i][0], test_set[i][1]
    im = np.einsum('ijk -> jki', im)

    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


for i in range(10, 20):
    im, label = test_set[glasses_test_count + i][0], test_set[glasses_test_count + i][1]
    im = np.einsum('ijk -> jki', im)


    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')

In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

for i in range(10):
    im, label = recons[i], test_set[i][1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


for i in range(10, 20):
    im, label = recons[100 + i], test_set[glasses_test_count + i][1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')

## CCVAE

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

ccvae = CNNCCVAE((64,64), 3, [1], latent_dim=2048, w_dim=2, channels=[64,128,256], repeats=[2,1,1], cnn_arch='conv+pool', num_layers=0, recon_weight=1e5, kl_weight=1e-4, kernel_size=5)
ccvae_trainer = ThresholdPyroTrainer(0, 50, ccvae, train_loader, test_loader, opt.AdamW({"lr": 1e-4}))
ccvae_trainer.train()

In [ ]:
preds_glasses = ccvae_trainer.predictive(*ccvae_trainer._send_args_to_device(ccvae_trainer.test_loader.dataset[:100], ccvae_trainer.device))
z_s_glasses = preds_glasses['z'][0].cpu()
w_s_glasses = preds_glasses['w'][0].cpu()
recons_glasses = preds_glasses['rec'][0, 0].cpu()


preds_ng = ccvae_trainer.predictive(*ccvae_trainer._send_args_to_device(ccvae_trainer.test_loader.dataset[glasses_test_count:glasses_test_count+100], ccvae_trainer.device))
z_s_ng = preds_ng['z'][0].cpu() 
w_s_ng = preds_ng['w'][0].cpu() 
recons_ng = preds_ng['rec'][0, 0].cpu()

z_s = torch.vstack((z_s_glasses, z_s_ng))
w_s = torch.vstack((w_s_glasses, w_s_ng))
recons = torch.vstack((recons_glasses, recons_ng))
y_s = torch.vstack((ccvae_trainer.test_loader.dataset[:100][1], ccvae_trainer.test_loader.dataset[glasses_test_count:glasses_test_count+100][1]))

In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

for i in range(10):
    im, label = test_set[i][0], test_set[i][1]
    im = np.einsum('ijk -> jki', im)

    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


for i in range(10, 20):
    im, label = test_set[glasses_test_count + i][0], test_set[glasses_test_count + i][1]
    im = np.einsum('ijk -> jki', im)


    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')

In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

for i in range(10):
    im, label = recons[i], test_set[i][1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


for i in range(10, 20):
    im, label = recons[100 + i], test_set[glasses_test_count + i][1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


## DISCoVeR

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

dlvae = CNNDLVAE((64,64), 3, [1], latent_dim=2048, w_dim=2048, channels=[64,128,256], repeats=[2,1,1], cnn_arch='conv+pool', num_layers=0, kernel_size=5, recon_weight=1e6, recon_weight_z=1e5, w_kl_weight=1e-4, z_kl_weight=1e-4, adversarial_weight=2e3, learnable_prior=False)
dlvae_trainer = AdversarialThresholdPyroTrainer(0, 50, 1, 2, dlvae, train_loader, test_loader, opt.AdamW({"lr": 1e-4}))
dlvae_trainer.train()

In [ ]:
window=0
samp=100

In [ ]:
preds_glasses = dlvae_trainer.predictive(*dlvae_trainer._send_args_to_device(dlvae_trainer.test_loader.dataset[window:samp+window], dlvae_trainer.device))
z_s_glasses = preds_glasses['z'][0].cpu()
w_s_glasses = preds_glasses['w'][0].cpu()
recons_w_glasses = preds_glasses['rec_w'][0, 0].cpu()
recons_z_glasses = preds_glasses['rec_z'][0, 0].cpu()


preds_ng = dlvae_trainer.predictive(*dlvae_trainer._send_args_to_device(dlvae_trainer.test_loader.dataset[glasses_test_count+window:glasses_test_count+window+samp], dlvae_trainer.device))
z_s_ng = preds_ng['z'][0].cpu() 
w_s_ng = preds_ng['w'][0].cpu() 
recons_w_ng = preds_ng['rec_w'][0, 0].cpu()
recons_z_ng = preds_ng['rec_z'][0, 0].cpu()


z_s = torch.vstack((z_s_glasses, z_s_ng))
w_s = torch.vstack((w_s_glasses, w_s_ng))
recons_w = torch.vstack((recons_w_glasses, recons_w_ng))
recons_z = torch.vstack((recons_z_glasses, recons_z_ng))
y_s = torch.vstack((dlvae_trainer.test_loader.dataset[window:window+samp][1], dlvae_trainer.test_loader.dataset[glasses_test_count+window:glasses_test_count+window+samp][1]))

In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

for i in range(10):
    im, label = test_set[i+window][0], test_set[i+window][1]
    im = np.einsum('ijk -> jki', im)

    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


for i in range(10, 20):
    im, label = test_set[glasses_test_count+window + i][0], test_set[glasses_test_count+window + i][1]
    im = np.einsum('ijk -> jki', im)


    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')

In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

for i in range(10):
    im, label = recons_w[i], test_set[i][1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


for i in range(10, 20):
    im, label = recons_w[100 + i], test_set[glasses_test_count + i][1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')

In [ ]:
fig, ax = plt.subplots(2,10, figsize=(15,3), gridspec_kw = {'wspace':0, 'hspace':-0.02})

for i in range(10):
    im, label = recons_z[i], test_set[i][1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)

    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')


for i in range(10, 20):
    im, label = recons_z[100 + i], test_set[glasses_test_count + i][1]
    im = torch.clamp(im, 0, 1)
    im = np.einsum('ijk -> jki', im)


    ax[i//10][i%10].imshow(im)
    ax[i//10][i%10].axis('off')